<h3>1. Baselines, Tuning, and OOF Meta-Training</h3>

<p>Tune and train H1/H2 development models using the Task 1 supervised tables and explicit chronological folds. The final test rows are never used for tuning or out-of-fold meta-training.</p>

In [ ]:
import base64
import os
from pathlib import Path
import shutil
import subprocess
import sys

import joblib
import pandas as pd

try:
    import google.colab  # type: ignore
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
    from dotenv import load_dotenv

REPO_URL = 'https://github.com/RAM-AS-RAP/ucs35-mudibo-ml.git'
REPO_BRANCH = 'feature/model-training'

if IN_COLAB:
    drive.mount('/content/drive')
    ENV_PATHS = [
        Path('/content/drive/MyDrive/capstone-data/.env'),
        Path('/content/drive/MyDrive/AgriSence/.env'),
        Path('/content/drive/MyDrive/.env'),
        Path('/content/.env'),
    ]
    for env_path in ENV_PATHS:
        if env_path.exists():
            load_dotenv(env_path, override=False)

    github_token = os.getenv('GITHUB_TOKEN') or os.getenv('GH_TOKEN')
    try:
        from google.colab import userdata
        github_token = github_token or userdata.get('GITHUB_TOKEN') or userdata.get('GH_TOKEN')
    except Exception:
        pass
else:
    ENV_PATHS = []
    github_token = None

if IN_COLAB:
    repo_dir = Path('/content/AgriSence_model_training')
    repo_candidates = [repo_dir]
else:
    repo_dir = None
    repo_candidates = [Path.cwd(), *Path.cwd().resolve().parents]

repo_root = next(
    (
        path.resolve() for path in repo_candidates
        if path.exists()
        and (path / 'pyproject.toml').exists()
        and (path / 'src' / 'agrisence').exists()
    ),
    None,
)

if repo_root is None and IN_COLAB:
    if not github_token:
        searched_paths = '\\n'.join(f'- {path}' for path in ENV_PATHS)
        raise RuntimeError(
            'Private repository clone requires GITHUB_TOKEN or GH_TOKEN. '
            'Add it to Colab Secrets (with notebook access enabled), or to a Drive .env file. '
            'Secrets could not be read in this runtime. Checked:\\n'
            f'{searched_paths}'
        )
    auth_header = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
    subprocess.run(
        [
            'git', '-c',
            f'http.https://github.com/.extraheader=AUTHORIZATION: basic {auth_header}',
            'clone', '-b', REPO_BRANCH, REPO_URL, str(repo_dir),
        ],
        check=True,
    )
    repo_root = repo_dir.resolve()

if repo_root is None:
    raise FileNotFoundError('Start this notebook from inside the AgriSence repository.')

if IN_COLAB:
    os.chdir(repo_root)

load_dotenv(repo_root / '.env', override=False)
sys.path.insert(0, str(repo_root / 'src'))

from agrisence.training.baselines.persistence import predict_persistence
from agrisence.training.baselines.linear_regression import fit_linear_regression
from agrisence.training.ensemble.random_forest import tune_random_forest
from agrisence.training.ensemble.xgboost_model import tune_xgboost
from agrisence.training.ensemble.stacking import generate_oof_predictions, fit_meta_model
from agrisence.training.folds import build_expanding_window_folds

if IN_COLAB:
    drive_artifact_dir = Path('/content/drive/MyDrive/capstone-data/feature_engineering')
    repo_artifact_dir = repo_root / 'results/artifacts/feature_engineering'
    artifact_dir = drive_artifact_dir if (drive_artifact_dir / 'model_df_h1.csv').exists() else repo_artifact_dir
    drive_training_artifacts = Path('/content/drive/MyDrive/capstone-data/model_training')
else:
    artifact_dir = repo_root / 'results/artifacts/feature_engineering'
    drive_training_artifacts = None

training_artifacts = repo_root / 'results/artifacts/model_training'
training_artifacts.mkdir(parents=True, exist_ok=True)

print(f'Repository: {repo_root}')
print(f'Active branch target: {REPO_BRANCH}')
print(f'Task 1 artifacts: {artifact_dir}')
print(f'Local training artifacts: {training_artifacts}')
if IN_COLAB:
    print(f'Drive training artifacts: {drive_training_artifacts}')

Mounted at /content/drive
Repository: /content/AgriSence_model_training
Active branch target: feature/model-training
Task 1 artifacts: /content/drive/MyDrive/capstone-data/feature_engineering


<h3>2. Train each horizon separately</h3>

<p>Feature columns are the Task 1 numeric log-return/ratio/calendar/identity columns. The development subset is used for tuning and OOF generation; the final test subset is loaded only to report its row count.</p>

In [2]:
import json
import numpy as np

DEVELOPMENT_METRIC_SCOPE = 'development-period chronological validation; not final test performance'

In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

from agrisence.training.folds import folds_to_sklearn_cv

feature_prefixes = ('log_return_lag_', 'log_ratio_', 'month_', 'market_', 'commodity_')
feature_cols = [
    column for column in pd.read_csv(artifact_dir / 'model_df_h1.csv', nrows=1).columns
    if column.startswith(feature_prefixes) or column == 'year'
]


def regression_metrics(actual, predicted):
    return {
        'mae': float(mean_absolute_error(actual, predicted)),
        'rmse': float(mean_squared_error(actual, predicted) ** 0.5),
    }


def mean_fold_metrics(fold_results):
    return {
        metric: float(np.mean([result[metric] for result in fold_results]))
        for metric in ('mae', 'rmse')
    }


results = {}
for horizon in ('h1', 'h2'):
    model_df = pd.read_csv(
        artifact_dir / f'model_df_{horizon}.csv',
        parse_dates=['month', 'target_month'],
    )
    folds = build_expanding_window_folds(model_df)
    development_df = model_df.loc[folds['development_mask']].reset_index(drop=True)
    test_df = model_df.loc[folds['test_mask']].reset_index(drop=True)
    target_col = f'target_{horizon}'
    cv_splits = folds_to_sklearn_cv(folds, development_df)

    persistence_fold_scores = []
    linear_fold_scores = []
    for train_indices, validation_indices in cv_splits:
        train_rows = development_df.iloc[train_indices]
        validation_rows = development_df.iloc[validation_indices]
        validation_target = validation_rows[target_col]

        persistence_fold_scores.append(
            regression_metrics(validation_target, predict_persistence(validation_rows))
        )
        fold_linear = fit_linear_regression(train_rows[feature_cols], train_rows[target_col])
        linear_fold_scores.append(
            regression_metrics(
                validation_target,
                fold_linear.predict(validation_rows[feature_cols]),
            )
        )

    persistence_metrics = mean_fold_metrics(persistence_fold_scores)
    linear_metrics = mean_fold_metrics(linear_fold_scores)
    linear_model = fit_linear_regression(development_df[feature_cols], development_df[target_col])
    rf_tuning = tune_random_forest(development_df, folds, feature_cols, target_col)
    xgb_tuning = tune_xgboost(development_df, folds, feature_cols, target_col)

    rf_params = rf_tuning['best_params']
    xgb_params = xgb_tuning['best_params']
    oof_df = generate_oof_predictions(
        development_df, folds, feature_cols, target_col, rf_params, xgb_params
    )
    rf_oof_metrics = regression_metrics(oof_df['target'], oof_df['rf_oof_pred'])
    xgb_oof_metrics = regression_metrics(oof_df['target'], oof_df['xgb_oof_pred'])
    base_prediction_correlation = float(
        oof_df[['rf_oof_pred', 'xgb_oof_pred']].corr(method='pearson').iloc[0, 1]
    )
    meta_model = fit_meta_model(oof_df)
    ridge_oof_fit_metrics = regression_metrics(
        oof_df['target'],
        meta_model.predict(oof_df[['rf_oof_pred', 'xgb_oof_pred']]),
    )

    development_metrics = {
        'scope': 'development-period chronological validation; not final test performance',
        'horizon': horizon,
        'folds': len(cv_splits),
        'persistence': persistence_metrics,
        'linear_regression': linear_metrics,
        'random_forest_oof': rf_oof_metrics,
        'xgboost_oof': xgb_oof_metrics,
        'rf_xgb_oof_prediction_pearson_correlation': base_prediction_correlation,
        'ridge_meta_oof_fit': ridge_oof_fit_metrics,
        'ridge_meta_oof_fit_note': 'Ridge is scored on the same OOF rows used to fit it; this is a fit diagnostic, not an independent meta-model validation score.',
        'random_forest_tuning_neg_mse': rf_tuning['best_score'],
        'xgboost_tuning_neg_mse': xgb_tuning['best_score'],
        'ridge_alpha': meta_model.best_alpha_,
        'development_rows': len(development_df),
        'held_out_test_rows_not_scored': len(test_df),
    }

    results[horizon] = {
        'feature_cols': feature_cols,
        'development_rows': len(development_df),
        'test_rows': len(test_df),
        'persistence_metrics': persistence_metrics,
        'linear_metrics': linear_metrics,
        'rf_tuning': rf_tuning,
        'xgb_tuning': xgb_tuning,
        'rf_oof_metrics': rf_oof_metrics,
        'xgb_oof_metrics': xgb_oof_metrics,
        'base_prediction_correlation': base_prediction_correlation,
        'oof_rows': len(oof_df),
        'meta_model': meta_model,
        'linear_model': linear_model,
        'development_metrics': development_metrics,
    }

    print(f'\\n{horizon.upper()} development-period results (not final test):')
    print('Persistence:', persistence_metrics)
    print('Linear Regression:', linear_metrics)
    print('Random Forest OOF:', rf_oof_metrics)
    print('XGBoost OOF:', xgb_oof_metrics)
    print('RF/XGBoost OOF prediction correlation:', base_prediction_correlation)
    print('Ridge fit on OOF rows:', ridge_oof_fit_metrics)
    print('Ridge alpha:', meta_model.best_alpha_)
    print('Test rows reserved and not scored:', len(test_df))

\nH1 development-period results (not final test):
Persistence: {'mae': 0.05921435103321375, 'rmse': 0.07987805512621304}
Linear Regression: {'mae': 0.05837086235376174, 'rmse': 0.07690730302612046}
Random Forest OOF: {'mae': 0.056912882750836814, 'rmse': 0.07661920075147728}
XGBoost OOF: {'mae': 0.05698560932975656, 'rmse': 0.07652379568170814}
RF/XGBoost OOF prediction correlation: 0.9269514287823221
Ridge fit on OOF rows: {'mae': 0.05620893055031456, 'rmse': 0.07606346097315238}
Ridge alpha: 0.1
Test rows reserved and not scored: 88
\nH2 development-period results (not final test):
Persistence: {'mae': 0.09288242731814954, 'rmse': 0.12113961678629202}
Linear Regression: {'mae': 0.09175225138314053, 'rmse': 0.11402941408620418}
Random Forest OOF: {'mae': 0.08855860655579467, 'rmse': 0.11411413572333143}
XGBoost OOF: {'mae': 0.09050132494017239, 'rmse': 0.11628821873091312}
RF/XGBoost OOF prediction correlation: 0.8933662908254028
Ridge fit on OOF rows: {'mae': 0.0877620154533599, 'rms

In [9]:
def format_selected_parameters(model_name, params=None):
    if model_name == 'Persistence':
        return 'Predict zero log change; no fitted parameters'
    if model_name == 'Linear Regression':
        return 'StandardScaler + LinearRegression; fixed baseline'
    if model_name == 'Random Forest':
        return (
            f"{params['n_estimators']} trees; depth {params['max_depth']}; "
            f"minimum leaf {params['min_samples_leaf']}; "
            f"max features {params['max_features']}"
        )
    if model_name == 'XGBoost':
        return (
            f"{params['n_estimators']} estimators; depth {params['max_depth']}; "
            f"learning rate {params['learning_rate']}; child weight {params['min_child_weight']}; "
            f"subsample {params['subsample']}; column sample {params['colsample_bytree']}; "
            f"L2 {params['reg_lambda']}"
        )
    return f"alpha {params['alpha']}"


tuning_rows = []
for horizon, result in results.items():
    models = [
        ('Persistence', None, None),
        ('Linear Regression', None, None),
        ('Random Forest', result['rf_tuning']['best_params'], -result['rf_tuning']['best_score']),
        ('XGBoost', result['xgb_tuning']['best_params'], -result['xgb_tuning']['best_score']),
        (
            'Ridge meta-model',
            {'alpha': result['meta_model'].best_alpha_},
            -result['meta_model'].best_score_,
        ),
    ]
    for model_name, params, cv_mse in models:
        tuning_rows.append({
            'Horizon': horizon.upper(),
            'Model': model_name,
            'Selected parameters': format_selected_parameters(model_name, params),
            'Development CV MSE': f'{cv_mse:.6f}' if cv_mse is not None else 'Not tuned',
        })

tuning_summary = pd.DataFrame(tuning_rows)
display(tuning_summary)

,Horizon,Model,Selected parameters,Development CV MSE
0,H1,Persistence,Predict zero log change; no fitted parameters,Not tuned
1,H1,Linear Regression,StandardScaler + LinearRegression; fixed baseline,Not tuned
2,H1,Random Forest,400 trees; depth 6; minimum leaf 10; max featu...,0.005963
3,H1,XGBoost,200 estimators; depth 3; learning rate 0.01; c...,0.005950
4,H1,Ridge meta-model,alpha 0.1,0.006284
5,H2,Persistence,Predict zero log change; no fitted parameters,Not tuned
6,H2,Linear Regression,StandardScaler + LinearRegression; fixed baseline,Not tuned
7,H2,Random Forest,200 trees; depth 6; minimum leaf 5; max featur...,0.013267
8,H2,XGBoost,200 estimators; depth 3; learning rate 0.01; c...,0.013694
9,H2,Ridge meta-model,alpha 1.0,0.014974


<h3>3. Compare development-period results</h3>


<p>These are development-only values. Persistence and Linear Regression are averaged across the same expanding validation folds; Random Forest and XGBoost use chronological OOF rows; Ridge is shown on the same OOF rows used to fit it, so its score is a fit diagnostic rather than an independent estimate.</p>

In [4]:
development_comparison_rows = []
for horizon, result in results.items():
    development_comparison_rows.extend([
        {'horizon': horizon, 'model': 'Persistence', **result['persistence_metrics']},
        {'horizon': horizon, 'model': 'Linear Regression', **result['linear_metrics']},
        {'horizon': horizon, 'model': 'Random Forest (OOF)', **result['rf_oof_metrics']},
        {'horizon': horizon, 'model': 'XGBoost (OOF)', **result['xgb_oof_metrics']},
        {
            'horizon': horizon,
            'model': 'Ridge stack (fit on OOF)',
            **result['development_metrics']['ridge_meta_oof_fit'],
        },
    ])
development_comparison = pd.DataFrame(development_comparison_rows)
display(development_comparison)
for horizon, result in results.items():
    print(
        horizon,
        'RF best CV neg-MSE=', result['rf_tuning']['best_score'],
        'XGBoost best CV neg-MSE=', result['xgb_tuning']['best_score'],
        'Ridge best CV neg-MSE=', result['meta_model'].best_score_,
    )

,horizon,model,mae,rmse
0,h1,Persistence,0.059214,0.079878
1,h1,Linear Regression,0.058371,0.076907
2,h1,Random Forest (OOF),0.056913,0.076619
3,h1,XGBoost (OOF),0.056986,0.076524
4,h1,Ridge stack (fit on OOF),0.056209,0.076063
5,h2,Persistence,0.092882,0.121140
6,h2,Linear Regression,0.091752,0.114029
7,h2,Random Forest (OOF),0.088559,0.114114
8,h2,XGBoost (OOF),0.090501,0.116288
9,h2,Ridge stack (fit on OOF),0.087762,0.114869


h1 RF best CV neg-MSE= -0.005962772191965723 XGBoost best CV neg-MSE= -0.005950363577639335 Ridge best CV neg-MSE= -0.006284267927072705
h2 RF best CV neg-MSE= -0.013266763967387102 XGBoost best CV neg-MSE= -0.013693835196684987 Ridge best CV neg-MSE= -0.014974192365306872


<h3>4. Save development metrics and models</h3>


<p>Save tuned hyperparameters, fold-averaged persistence/linear metrics, OOF base/stack metrics, and the fitted Ridge meta-model for the next task. These are development-only results; final test rows are not used for tuning, OOF generation, or these summaries.</p>

In [11]:
import shutil

if IN_COLAB:
    drive_training_artifacts = Path('/content/drive/MyDrive/capstone-data/model_training')

created_artifacts = []
for horizon, result in results.items():
    tuning_payload = {
        'horizon': horizon,
        'feature_cols': result['feature_cols'],
        'development_rows': result['development_rows'],
        'test_rows_reserved_not_scored': result['test_rows'],
        'random_forest': result['rf_tuning'],
        'xgboost': result['xgb_tuning'],
        'ridge_meta': {
            'alpha': result['meta_model'].best_alpha_,
            'cv_score': result['meta_model'].best_score_,
        },
        'oof_rows': result['oof_rows'],
    }
    tuning_path = training_artifacts / f'tuning_{horizon}.json'
    persistence_path = training_artifacts / f'persistence_dev_metrics_{horizon}.json'
    linear_metrics_path = training_artifacts / f'linreg_dev_metrics_{horizon}.json'
    oof_metrics_path = training_artifacts / f'oof_metrics_{horizon}.json'
    linear_model_path = training_artifacts / f'linear_regression_{horizon}.joblib'
    ridge_model_path = training_artifacts / f'ridge_meta_{horizon}.joblib'

    tuning_path.write_text(json.dumps(tuning_payload, indent=2))
    persistence_path.write_text(json.dumps({
        'scope': DEVELOPMENT_METRIC_SCOPE,
        **result['persistence_metrics'],
    }, indent=2))
    linear_metrics_path.write_text(json.dumps({
        'scope': DEVELOPMENT_METRIC_SCOPE,
        **result['linear_metrics'],
    }, indent=2))
    oof_metrics_path.write_text(json.dumps(result['development_metrics'], indent=2))
    joblib.dump(result['linear_model'], linear_model_path)
    joblib.dump(result['meta_model'], ridge_model_path)

    created_artifacts.extend([
        tuning_path,
        persistence_path,
        linear_metrics_path,
        oof_metrics_path,
        linear_model_path,
        ridge_model_path,
    ])
    print(f'Saved local development artifacts for {horizon}: {training_artifacts}')

if IN_COLAB:
    drive_training_artifacts.mkdir(parents=True, exist_ok=True)
    for local_path in created_artifacts:
        drive_path = drive_training_artifacts / local_path.name
        shutil.copy2(local_path, drive_path)
        print(f'Mirrored to Google Drive: {drive_path} (exists={drive_path.exists()})')
else:
    print('Google Drive mirror skipped: notebook is running outside Colab.')

Saved local development artifacts for h1: /content/AgriSence_model_training/results/artifacts/model_training
Saved local development artifacts for h2: /content/AgriSence_model_training/results/artifacts/model_training
Mirrored to Google Drive: /content/drive/MyDrive/capstone-data/model_training/tuning_h1.json (exists=True)
Mirrored to Google Drive: /content/drive/MyDrive/capstone-data/model_training/persistence_dev_metrics_h1.json (exists=True)
Mirrored to Google Drive: /content/drive/MyDrive/capstone-data/model_training/linreg_dev_metrics_h1.json (exists=True)
Mirrored to Google Drive: /content/drive/MyDrive/capstone-data/model_training/oof_metrics_h1.json (exists=True)
Mirrored to Google Drive: /content/drive/MyDrive/capstone-data/model_training/linear_regression_h1.joblib (exists=True)
Mirrored to Google Drive: /content/drive/MyDrive/capstone-data/model_training/ridge_meta_h1.joblib (exists=True)
Mirrored to Google Drive: /content/drive/MyDrive/capstone-data/model_training/tuning_h2